# Rodar experimentos no Colab

Este notebook roda as famílias de `config/runs/` numa GPU do Google Colab, com o mesmo `train.py` e o mesmo
`evaluate.py` do repositório. Ele só **executa**: as configs vêm do GitHub e os resultados voltam para `runs/`.

[Abrir no Colab](https://colab.research.google.com/github/joaosaraivarodrigues1/NEODM/blob/experimentos/colab.ipynb)

1. **Ambiente de execução → Alterar o tipo de ambiente → T4 GPU.**
2. **Executar tudo** e deixar a aba aberta.
3. Cada run que termina é copiada para o Google Drive (`MyDrive/NEODM-runs`). Se a sessão cair,
   é só executar tudo de novo: o notebook recupera o que já está no Drive e continua de onde parou.
4. No fim, baixar a pasta do Drive e extrair as pastas novas em `runs/` do repositório local.

## 1. Clonar o repositório

In [ ]:
import os
import re
import json
import time
import shutil
import hashlib
import subprocess

REPO = 'https://github.com/joaosaraivarodrigues1/NEODM.git'
BRANCH = 'experimentos'

if not os.path.exists('/content/NEODM'):
    !git clone -q -b {BRANCH} {REPO} /content/NEODM
%cd /content/NEODM
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Google Drive

As runs terminadas ficam no Drive. Esta célula traz de volta para `runs/` as runs de uma sessão anterior.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PASTA_DRIVE = '/content/drive/MyDrive/NEODM-runs'
os.makedirs(PASTA_DRIVE, exist_ok=True)

for run in sorted(os.listdir(PASTA_DRIVE)):
    if not os.path.exists(f'runs/{run}'):
        shutil.copytree(f'{PASTA_DRIVE}/{run}', f'runs/{run}')
        print('recuperada do Drive:', run)

## 3. Dados

Os dados não sobem para o GitHub: o gerador usa a seed 1337 e sai sempre igual.
Os checksums conferem que os `.bin` gerados aqui são idênticos aos do computador local.

In [ ]:
if not os.path.exists('data/datas/test.bin'):
    !python data/datas/gerar_dados.py
    !python data/datas/prepare.py

CHECKSUMS = {
    'train': '9102fc1293d0583c2e8d3448c8cb5c1c',
    'val': 'f911013fdbb6eccc86db1bb269459028',
    'test': 'b00f99e4b69ae6bb994535c2ee032588',
}
for split, md5 in CHECKSUMS.items():
    with open(f'data/datas/{split}.bin', 'rb') as f:
        assert hashlib.md5(f.read()).hexdigest() == md5, f'{split}.bin diferente do computador local'
print('dados idênticos aos do computador local')

## 4. Rodar uma run

A mesma regra do `rodar.ps1`: cria `runs/<familia>_s<N>` com o próximo N livre e a seed 1336 + N,
treina, avalia no teste e copia a pasta para o Drive.

In [ ]:
def completas(familia):
    """Runs da família que terminaram: as que têm resultado_test.json."""
    return [r for r in os.listdir('runs')
            if re.fullmatch(rf'{familia}_s\d+', r) and os.path.exists(f'runs/{r}/resultado_test.json')]

def rodar(familia):
    n = 1
    while os.path.exists(f'runs/{familia}_s{n}'):
        n += 1
    seed, out = 1336 + n, f'runs/{familia}_s{n}'
    print(f'== {out} (seed {seed})', flush=True)
    inicio = time.time()
    comandos = [
        ['python', 'train.py', 'config/datas.py', f'config/runs/{familia}.py', f'--seed={seed}', f'--out_dir={out}'],
        ['python', 'evaluate.py', f'--out_dir={out}'],
    ]
    for cmd in comandos:
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:], r.stderr[-2000:])
            shutil.rmtree(out, ignore_errors=True)   # não deixa run incompleta
            print(f'   falhou: {out} apagada')
            return False
    shutil.copytree(out, f'{PASTA_DRIVE}/{familia}_s{n}', dirs_exist_ok=True)
    with open(f'{out}/resultado_test.json', encoding='utf-8') as f:
        res = json.load(f)
    print(f"   comum {100 * res['por_parte']['comum']:.2f}% | {res['config']['gpu']} | {(time.time() - inicio) / 60:.1f} min", flush=True)
    return True

## 5. A leva

Completa cada família até 3 runs. As runs que já existem (no GitHub ou no Drive) contam:
executar de novo nunca duplica. Os modelos maiores ficam por último.

In [ ]:
FAMILIAS = [
    'out_colab_base',                                   # checagem: a base na T4, para comparar com a MX550
    'abl_relu', 'abl_sem_init_cproj', 'abl_sem_wd',     # ablações que faltam
    'abl_dropout01', 'abl_lr_constante', 'abl_sem_clip',
    'par_raso_L1', 'par_fundo_L4',                      # profundidade, com o mesmo número de parâmetros
    'par_cabecas1', 'par_cabecas2', 'par_cabecas8',     # cabeças de atenção
    'par_modelo_08M', 'par_modelo_48M',                 # escala
]
SEEDS = 3

# uma run interrompida no meio (sem resultado) é apagada, para o N não pular
for run in sorted(os.listdir('runs')):
    if not os.path.exists(f'runs/{run}/resultado_test.json'):
        shutil.rmtree(f'runs/{run}')
        print('incompleta apagada:', run)

for familia in FAMILIAS:
    while len(completas(familia)) < SEEDS:
        if not rodar(familia):
            break   # erro: passa para a próxima família
print('fim da leva')

## 6. Resumo das runs no Drive

In [ ]:
import pandas as pd

linhas = []
for run in sorted(os.listdir(PASTA_DRIVE)):
    with open(f'{PASTA_DRIVE}/{run}/resultado_test.json', encoding='utf-8') as f:
        r = json.load(f)
    linha = {'run': run, 'gpu': r['config'].get('gpu'), 'git_commit': r['config'].get('git_commit')}
    linha.update({parte: 100 * v for parte, v in r['por_parte'].items()})
    linhas.append(linha)
pd.DataFrame(linhas).set_index('run').round(2)

## 7. Levar para o repositório local

1. No Google Drive, baixar a pasta `NEODM-runs` (vem em zip).
2. Extrair as pastas de dentro dela em `NEODM/runs/`.
3. Rodar o `runs.ipynb` e fazer o commit. O `ckpt.pt` e o `erros_test.csv` ficam fora do git pelo `.gitignore`.